In [1]:
import sys
sys.path.append('../src')

from linguistic_brain_toolkit.utilities import get_path
PATH = get_path(1)

In [2]:
import sqlite3
import pandas as pd

from linguistic_brain_toolkit.data import read_data_from_database, load_dataframe_to_sqlite
from linguistic_brain_toolkit.nlp import apply_lemmatization_and_pos_tags
from linguistic_brain_toolkit.utilities import get_path

pd.set_option('display.precision', 15)  # Establecer la precisión a 15 decimales

PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/processed/transcriptions.db'

df = read_data_from_database(DATABASE_FOLDER, 'general')

df.head()

,start,stop,name,tier,source
0,0.0000000000000,0.0124716553288,,phone,adollshouse
1,0.0124716553288,0.3816326530610,sp,phone,adollshouse
2,0.3816326530610,0.4215419501130,AO2,phone,adollshouse
3,0.4215419501130,0.4514739229020,L,phone,adollshouse
4,0.4514739229020,0.5213151927440,R,phone,adollshouse


In [3]:
# Preprocesing
df['name'] = df['name'].str.lower()
df['tier'].value_counts()

df_graphemic_transcrip = df.query("tier == 'word'").copy()

# Filtrar el DataFrame para excluir las filas que contienen las cadenas buscadas
strings_to_exclude = ('{', 'sp', 'chuck-e-cheese')

condition = df_graphemic_transcrip.astype(str).apply(
    lambda row: row.str.contains('|'.join(strings_to_exclude)
                                 )).any(axis=1)
df_filtered = df_graphemic_transcrip[~condition].copy()

# df_filtered = df_filtered.sample(n=5000)  #AAA

In [4]:
# NLP
df_processed = apply_lemmatization_and_pos_tags(df_filtered, 'name', 'large')
del df, df_graphemic_transcrip, df_filtered  # Delete dataframes to free memory
df_processed.head()

,start,stop,name,tier,source,lemmatized_text,pos_tags,pos_tag_1,pos_tag_2,pos_tag_3
5412,0.000000000000,0.0124716553288,,word,adollshouse,,[],None,None,None
5414,0.381632653061,0.6609977324260,alright,word,adollshouse,alright,[INTJ],INTJ,None,None
5415,0.660997732426,0.9104308390020,thank,word,adollshouse,thank,[VERB],VERB,None,None
5416,0.910430839002,1.0102040816300,you,word,adollshouse,you,[PRON],PRON,None,None
5417,1.010204081630,1.2496598639500,very,word,adollshouse,very,[ADV],ADV,None,None


In [5]:
df_processed['pos_tag_1'].value_counts()

pos_tag_1
PRON     40039
VERB     26664
NOUN     19446
ADP      16013
ADV      10712
CCONJ     9212
AUX       7330
ADJ       5868
INTJ      4352
PROPN     3634
SCONJ     2039
NUM       2009
PART       501
X          100
PUNCT        8
DET          7
Name: count, dtype: int64

In [6]:
df_processed['pos_tag_2'].value_counts()

pos_tag_2
AUX      2733
PART     1935
VERB      353
PRON       30
ADP        24
PUNCT       6
NOUN        2
PROPN       1
ADJ         1
SPACE       1
Name: count, dtype: int64

In [7]:
df_processed['lemmatized_text'].value_counts()

lemmatized_text
I            8908
and          7850
the          5968
be           5788
to           3828
             ... 
jungle          1
holder          1
stoplight       1
weirdo          1
confusion       1
Name: count, Length: 7191, dtype: int64

In [8]:
cols_to_process = [col for col in df_processed.columns if col.startswith('pos_tag_')]

for col in cols_to_process:
    df_processed[col] = df_processed[col].fillna('')  # Manejar valores nulos

df_processed.reset_index(drop=True, inplace=True)
df_processed.drop(['pos_tags'], axis=1, inplace=True)  # Especificar axis=1 para eliminar columnas

load_dataframe_to_sqlite(
    df_processed, DATABASE_FOLDER, 'nlp_graphemic_transcriptions',
    if_exists='replace', df_name='df_processed'
    )

DataFrame df_processed loaded into table nlp_graphemic_transcriptions.
